# 8.10 Transformer 的三种不同架构：理解、生成与输入输出转换

jshn9515  
2026-05-08

<a href="https://colab.research.google.com/github/jshn9515/dnnl-notebooks/blob/main/zh/ch8-attention-and-transformer/ch8.10-transformer-architectures.ipynb" data-fig-align="left"><img src="https://colab.research.google.com/assets/colab-badge.svg" /></a>

前面几节里，我们已经分别看过 Transformer Encoder 和 Transformer Decoder。

- Encoder 的核心是 **self-attention**，它让序列中的每个 token 同时看到左右两边的上下文，因此很适合做理解任务；
- Decoder 的核心是 **masked self-attention**，它只能看到当前位置之前的 token，因此很适合做自回归生成任务。

原始 Transformer 是一个完整的 encoder-decoder 结构：encoder 先理解输入序列，decoder 再基于 encoder 输出一步一步生成目标序列。到这里，一个自然的问题是：

> **所有 Transformer 模型都必须同时有 encoder 和 decoder 吗？**

答案是否定的。在后来的模型发展中，Transformer 被拆成了几种常见架构：**Encoder-only**、**Decoder-only** 和 **Encoder-Decoder**。它们都来自 Transformer，但适合的任务和训练方式并不一样。

这一节，我们就把这三种结构放在一起比较一下。

## 8.10.1 从原始 Transformer 说起

原始 Transformer 是为机器翻译设计的。机器翻译有一个输入句子，也有一个输出句子。比如：

``` text
English: I love deep learning.
Chinese: 我喜欢深度学习。
```

这类任务天然可以分成两部分。第一部分是理解源语言句子：

$$x_1, x_2, \dots, x_m$$

第二部分是生成目标语言句子：

$$y_1, y_2, \dots, y_n$$

所以原始 Transformer 使用 encoder-decoder 架构。

Encoder 负责处理源句：

$$H = \operatorname{Encoder}(X)$$

Decoder 负责生成目标句：

$$p(y_t \mid y_{<t}, X) = \operatorname{Decoder}(y_{<t}, H)$$

这里的 $H$ 是 encoder 输出的上下文表示。

Decoder 在生成每个 token 时，不仅会看已经生成的目标 token，还会通过 cross-attention 去看源句表示。因此，完整的 Transformer 可以写成：

$$X \rightarrow \operatorname{Encoder} \rightarrow H \rightarrow \operatorname{Decoder} \rightarrow Y$$

如果任务是给一个输入，生成另一个输出，这种结构非常自然。但是后来人们发现，Transformer 的 encoder 和 decoder 本身也可以单独拿出来使用。

## 8.10.2 Encoder-only：更适合理解

Encoder-only 模型只保留 Transformer Encoder，不使用 decoder。它的结构可以简单写成：

$$X \rightarrow \mathrm{Transformer Encoder} \rightarrow H$$

其中 $H$ 是每个 token 的上下文表示：

$$H = [h_1, h_2, \dots, h_n]$$

Encoder-only 的关键特点是：

> **每个 token 都可以同时看见左右两边的上下文。**

也就是说，在 encoder self-attention 中，第 $i$ 个 token 可以关注到：

$$x_1, x_2, \dots, x_n$$

这既包括它左边的 token，也包括它右边的 token。因此，encoder-only 模型很适合做理解类任务，比如文本分类、情感分析、命名实体识别、句子匹配、抽取式问答和 embedding 表示学习。这些任务通常不需要模型一个 token 一个 token 地生成长文本，而是需要模型充分理解已有输入。

最典型的 encoder-only 模型就是 **BERT**。

BERT 的全称是 Bidirectional Encoder Representations from Transformers。这个名字已经说明了它的核心：它使用 Transformer Encoder 来学习双向上下文表示。

比如句子：

``` text
The animal did not cross the street because it was tired.
```

当模型处理 `it` 的时候，它可以同时看到前面的：

``` text
The animal did not cross the street because
```

也可以看到后面的：

``` text
was tired
```

这种双向上下文对理解任务非常有帮助。因为很多时候，一个词的含义需要结合左右两边的信息才能判断。

BERT 的预训练任务之一是 masked language modeling。比如把句子中的某些 token 替换成 `[MASK]`：

``` text
The animal did not cross the street because [MASK] was tired.
```

模型要根据左右上下文预测被 mask 掉的词。这和自回归语言模型不同。自回归语言模型只能根据左侧上下文预测下一个词：

$$p(x_t \mid x_{<t})$$

而 masked language modeling 更像是：

$$p(x_i \mid x_{\setminus i})$$

也就是根据除当前位置之外的上下文预测被遮住的 token。所以，BERT 不是用来从左到右生成文本的模型，它更擅长给已有文本做理解和表示。

那么，我们该如何使用 encoder-only 模型的输出呢？

我们知道，encoder-only 模型输出的是一串上下文表示：

$$h_1, h_2, \dots, h_n$$

不同任务会用这些表示的不同部分。对于文本分类，通常会取一个特殊 token 的表示，比如 BERT 里的 `[CLS]`：

$$h_{\mathrm{[CLS]}}$$

然后接一个分类头：

$$\hat{y} = \mathrm{Classifier}(h_{\mathrm{[CLS]}})$$

对于命名实体识别这类 token-level 任务，则会对每个 token 的表示分别分类：

$$\hat{y}_i = \mathrm{Classifier}(h_i)$$

对于句子 embedding，可以把 token 表示做 pooling，比如平均池化：

$$h_{\mathrm{avg}} = \frac{1}{n} \sum_{i=1}^{n} h_i$$

所以，encoder-only 模型的输出不是直接生成文本，而是提供一组高质量的上下文表示。后面接什么任务头，取决于具体任务。

## 8.10.3 Decoder-only：更适合生成

Decoder-only 模型只保留 Transformer Decoder 里的 masked self-attention 部分。不过要注意，由于模型是 decoder-only，所以通常没有 encoder，也就没有 cross-attention。它只使用 masked self-attention 和 feed-forward network。结构可以简单写成：

$$X \rightarrow \mathrm{Transformer Decoder Blocks} \rightarrow H
\rightarrow \mathrm{LM\ Head} \rightarrow p(x_{t+1})$$

Decoder-only 的关键特点是：

> **每个 token 只能看到自己之前的 token。**

也就是说，第 $i$ 个 token 只能观察到：

$$x_1, x_2, \dots, x_i$$

不能看到：

$$x_{i+1}, x_{i+2}, \dots, x_n$$

这个约束由 causal mask 实现。因此，decoder-only 模型天然适合自回归语言建模：

$$p(x_1, x_2, \dots, x_n) = \prod_{t=1}^{n} p(x_t \mid x_{<t})$$

也就是把整段文本的概率拆成一步一步的条件概率。最典型的 decoder-only 模型就是 GPT 系列。

GPT 的核心训练目标是 next-token prediction。给定一段文本：

``` text
I love deep learning
```

训练时模型会学习：

$$p(\text{love} \mid \text{I})$$

$$p(\text{deep} \mid \text{I love})$$

$$p(\text{learning} \mid \text{I love deep})$$

也就是说，模型始终根据前面的 token 预测下一个 token。这和 decoder-only 的结构完全匹配，因为 masked self-attention 保证了模型在预测某个位置时不能看到未来 token。

推理时，GPT 也按照同样的方式生成文本：

$$x_1 \rightarrow x_2 \rightarrow x_3 \rightarrow \cdots$$

每一步都根据已经生成的前缀预测下一个 token。所以，decoder-only 模型特别适合开放式生成任务，比如续写、对话、摘要生成、代码生成、问答生成和指令跟随。这些任务的共同点是：输出通常不是一个固定类别，而是一段需要逐步生成的文本。

那么，既然 encoder-only 更适合理解，decoder-only 更适合生成，那 decoder-only 能不能做理解任务？答案是可以。比如文本分类任务，我们可以把问题改写成生成任务。

原始分类形式可能是：

> 这句话的情感是正面还是负面？

Decoder-only 模型可以用 prompt 的形式处理：

    Sentence: I really like this movie.  
    Sentiment:

然后让模型生成 `positive`。

也就是说，decoder-only 模型可以把很多任务统一成“给定前缀，生成答案”。这也是现代大语言模型非常强大的原因之一。它不一定需要为每个任务单独设计分类头，而是可以通过自然语言 prompt 把不同任务都变成文本生成。

不过从结构上看，decoder-only 仍然是单向的。它处理输入时，每个位置只能看左侧上下文。只是当完整 prompt 都作为前缀输入时，模型在生成答案的位置可以看到整个 prompt。例如：

``` text
Question: What is the capital of France? Answer:
```

当模型生成 `Paris` 时，它已经可以观察到前面整个 question。因此，对答案位置来说，它确实看到了完整输入。这就是 decoder-only 模型用于理解任务的基本方式：不是让输入内部每个 token 双向交互，而是把任务组织成一个前缀，让答案在后面生成。

## 8.10.4 Encoder-Decoder：更适合输入到输出的转换

Encoder-decoder 模型同时使用 encoder 和 decoder。结构可以写成：

$$H = \operatorname{Encoder}(X)$$

$$p(y_t \mid y_{<t}, X) = \operatorname{Decoder}(y_{<t}, H)$$

它的特点是把理解输入和生成输出分开。Encoder 可以对输入序列做双向建模：

$$x_i \text{ can see } x_1, x_2, \dots, x_n$$

Decoder 则对输出序列做自回归建模：

$$y_t \text{ can only see } y_1, y_2, \dots, y_{t-1}$$

同时，decoder 通过 cross-attention 读取 encoder 的输出：

$$\operatorname{CrossAttention}(Q_{\mathrm{decoder}}, K_{\mathrm{encoder}}, V_{\mathrm{encoder}})$$

这种结构特别适合 seq2seq 任务，也就是输入和输出都是序列，但二者不一定长度相同。典型任务包括机器翻译、文本摘要、文本改写、语法纠错、文本到 SQL、文本到代码，以及多模态任务中的图像描述生成。典型模型包括原始 Transformer、T5、BART 等。

T5 是一个典型的 encoder-decoder 模型。它的核心思想是：把所有 NLP 任务都转换成 text-to-text 格式。

比如翻译任务可以写成：

``` text
Translate English to Chinese: That is good.
```

输出是：`那很好。`

摘要任务可以写成：

``` text
> Summarize: long article ...
```

输出是：`a short summary...`

分类任务也可以写成：

``` text
Sentiment: I love this movie.
```

输出是：`positive`。

这样一来，不管任务原来是什么形式，都可以统一成：

$$\text{text input} \rightarrow \text{text output}$$

这和 encoder-decoder 结构非常匹配。Encoder 负责理解输入文本，decoder 负责生成输出文本。

相比 decoder-only 模型，encoder-decoder 模型在处理“给定输入，生成输出”的任务时有一个结构上的优势：输入部分可以被 encoder 双向建模，而输出部分仍然由 decoder 自回归生成。也就是说，输入文本内部不需要遵守从左到右的生成限制，encoder 可以充分理解整个输入。

## 8.10.5 三种架构的核心区别

我们可以从 attention mask 的角度理解三种架构。Encoder-only 使用双向 self-attention：

$$x_i \rightarrow x_1, x_2, \dots, x_n$$

每个位置都可以看整个输入。

Decoder-only 使用 causal self-attention：

$$x_i \rightarrow x_1, x_2, \dots, x_i$$

每个位置只能看自己和之前的位置。

Encoder-decoder 同时使用两种 attention。Encoder 中是双向 self-attention：

$$x_i \rightarrow x_1, x_2, \dots, x_m$$

Decoder 中是 causal self-attention：

$$y_t \rightarrow y_1, y_2, \dots, y_t$$

Decoder 还会通过 cross-attention 看 encoder 输出：

$$y_t \rightarrow H_{\mathrm{encoder}}$$

所以，从结构上看，我们可以把三种架构总结成下面这个表格：

| 架构 | 能否双向看输入 | 典型模型 | 常见任务 |
|----|----|----|----|
| Encoder-only | 是 | BERT | 理解、分类、抽取、embedding |
| Decoder-only | 否，causal | GPT | 续写、对话、生成、指令跟随 |
| Encoder-Decoder | Encoder 双向，Decoder causal | Transformer、T5、BART | 翻译、摘要、输入到输出转换 |

表 1：三种 Transformer 架构的核心区别

这张表只是一个大致总结。现实中，不同模型还会有很多变体，但基本思想是一样的。

## 8.10.6 什么时候用哪种结构

如果任务主要是理解已有输入，并输出一个类别、标签或向量表示，通常采用 encoder-only。例如：

> 给一段文本，判断情感是正面还是负面。

这类任务可以用 BERT 风格模型，把输入编码成上下文表示，再接分类头。

如果任务是开放式生成，通常采用 decoder-only。例如：

> 给一个开头，续写一段文本。

这类任务本来就是从左到右生成，GPT 风格模型非常适合。

如果任务是明确的输入到输出转换，通常是 encoder-decoder。例如：

> 给一篇文章，生成摘要。  
> 给一句英文，翻译成中文。  
> 给一段自然语言问题，生成 SQL 查询。

这类任务有一个完整输入，也有一个需要生成的输出。Encoder 先双向理解输入，decoder 再基于输入自回归生成输出。

不过，在现代大模型里，这些边界并不绝对。Decoder-only 模型也可以通过 prompt 做分类、翻译、摘要；encoder-decoder 模型也可以做很多生成任务；encoder-only 模型也可以通过特殊设计用于检索或匹配。所以更准确地说，架构不是任务能力的唯一决定因素，它只是给某些任务提供了更自然的计算形式。

## 8.10.7 为什么现在 Decoder-only 很流行

从传统 NLP 任务看，encoder-only 和 encoder-decoder 都很自然。BERT 很适合理解任务，T5 很适合 text-to-text 任务。但近年来，大语言模型中 decoder-only 架构非常流行。

一个重要原因是：

> **Next-token prediction 是一个非常通用、非常容易扩展的训练目标。**

只要有大量文本，就可以训练模型预测下一个 token：

$$p(x_t \mid x_{<t})$$

这样，我们不需要为每个样本人工标注具体任务标签，也不需要把数据整理成特定输入输出格式。

同时，decoder-only 模型在推理时也很简单：输入一段文本，它接着往后写。很多任务都可以改写成这种形式。比如，问答就是输入问题，让模型生成答案；翻译就是输入翻译要求和原句，让模型生成译文；分类也可以写成“请判断下面文本属于哪个类别”，然后让模型生成类别名称；代码任务则可以输入需求描述或已有代码，让模型继续补全。

因此，虽然这些任务表面上很不一样，但在 decoder-only 模型看来，它们都可以归结为同一件事：

$$\text{prompt} \rightarrow \text{completion}$$

这种统一性非常适合大规模预训练和指令微调。当然，这并不意味着 decoder-only 在所有任务上都天然优于 encoder-only 或 encoder-decoder。架构选择仍然和任务形式、数据规模、训练方式、推理需求有关。

## 8.10.8 本章小结

这一节里，我们把 Transformer 的三种常见架构放在一起比较了一下。

Encoder-only 只使用 Transformer Encoder，核心是双向 self-attention。它让每个 token 都能看到整个输入序列，因此很适合文本理解、分类、抽取和表示学习。BERT 是最典型的 encoder-only 模型。

Decoder-only 只使用带 causal mask 的 Transformer Decoder，核心是从左到右的自回归建模。它通过 next-token prediction 学习语言分布，并在推理时一步一步生成文本。GPT 系列是最典型的 decoder-only 模型。

Encoder-decoder 同时使用 encoder 和 decoder。Encoder 双向理解输入，decoder 通过 masked self-attention 生成输出，并通过 cross-attention 读取 encoder 表示。它非常适合机器翻译、摘要、文本改写等输入到输出的转换任务。原始 Transformer、T5 和 BART 都属于这一类。

从更大的角度看，这三种架构不是彼此割裂的模型家族，而是 Transformer 组件的不同组合方式。理解它们的差异，可以帮助我们更清楚地理解为什么 BERT 擅长理解，GPT 擅长生成，而 T5/BART 擅长 text-to-text 转换。

到这里，第 8 章关于 attention 和 Transformer 的主线已经基本完整了。接下来我们继续往实现方向走，用 Hugging Face Transformers 加载真实模型，观察不同架构在代码接口、输入输出和 attention mask 上的差异。